# 11 — Next-Basket Prediction with MLflow

## 1. Objective

In Notebook 10, we created the final feature table for next-basket prediction.

Now we will train a machine learning model that predicts which products are most likely to appear in each customer's next basket.

In this notebook, we will:

- Load the final feature table
- Split customers into training and validation groups
- Train machine learning models
- Track each model with MLflow
- Compare their results
- Evaluate the final Top-N recommendations
- Select the best model

The final model will later be used to generate product recommendations for customers.

## 2. Create the MLflow Experiment

We will use MLflow to keep track of the models we train.

For each model, MLflow can save information such as:

- Model name
- Model settings
- Evaluation results
- Training run
- Final model

This will allow us to compare our models in Databricks instead of keeping the results only inside notebook cells.

In [0]:
import mlflow

experiment_name = "/Shared/instacart_next_basket_prediction"

experiment = mlflow.set_experiment(experiment_name)

print("MLflow experiment ready")
print("Experiment name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)

MLflow experiment ready
Experiment name: /Shared/instacart_next_basket_prediction
Experiment ID: 467959497818941


## 3. Load the Final Feature Table

We load the dataset created in Notebook 10.

This table contains the candidate products and the features that will be used to predict whether each product will appear in the customer's next basket.

In [0]:
from pyspark.sql import functions as F

features_df = spark.table(
    "workspace.ml_data.next_basket_features"
)

print("Dataset loaded")
print("Number of columns:", len(features_df.columns))

display(features_df.limit(5))

Dataset loaded
Number of columns: 84


user_id,product_id,department_id,aisle_id,target_order_id,is_reorder_candidate,is_aisle_candidate,is_copurchase_candidate,aisle_candidate_rank,copurchase_candidate_rank,candidate_source_count,candidate_source,target_purchased,customer_prior_orders,customer_total_products,customer_unique_products,customer_avg_basket_size,customer_reorder_rate,customer_avg_days_between_orders,customer_std_days_between_orders,customer_avg_order_hour,customer_active_days_of_week,customer_last_basket_size,customer_avg_last_3_basket_size,customer_basket_size_trend,customer_preferred_dow,customer_preferred_day_share,customer_preferred_hour,customer_preferred_hour_share,product_purchase_count,product_unique_customers,product_reorder_rate,product_avg_cart_position,product_name,product_purchases_per_customer,product_order_share,product_preferred_dow,product_preferred_day_share,product_preferred_hour,product_preferred_hour_share,user_product_order_count,user_product_reorder_rate,user_product_avg_cart_position,user_product_first_order,user_product_last_order,user_product_order_share,orders_since_last_product_purchase,user_product_avg_order_gap,user_product_due_score,bought_in_last_order,purchases_last_3_orders,purchases_last_5_orders,purchase_rate_last_3_orders,purchase_rate_last_5_orders,recent_purchase_momentum,user_product_max_streak,user_product_current_streak,is_new_to_customer,target_order_dow,target_order_hour,target_days_since_prior_order,order_timing_deviation_days,order_timing_ratio,expected_basket_size,recent_vs_usual_basket_ratio,target_order_number,has_order_timing_ratio,customer_aisle_affinity,customer_department_affinity,customer_day_match,product_day_match,customer_day_distance,product_day_distance,customer_hour_distance_circular,product_hour_distance_circular,customer_basket_stability,aisle_in_last_basket,department_in_last_basket,repeat_stability_signal,last_basket_context_level,stability_weighted_context,aisle_candidate_score,copurchase_candidate_score,has_reorder_cycle
159734,42768,15,59,1420022,1,0,0,0,0,1,reorder,0,50,1109,151,22.18,0.8638,7.24,2.35,13.64,7,39,36.33,14.15,0,0.5,9,0.18,19199,8436,0.5606,9.89,Organic Garbanzo Beans,2.28,0.009377,0,0.2339,11,0.0859,1,0.0,11.0,2,2,0.02,49,0.0,0.0,0,0,0,0.0,0.0,-0.02,1,0,0,6,20,8.0,0.76,1.1,29.26,1.64,51,1,0.0189,0.0397,0,0,1,1,11,9,0.4667,0,1,0.0,1,0.15556666666666666,0.0,0.0,0
116395,12206,9,9,478990,1,0,0,0,0,1,reorder,0,5,47,37,9.4,0.2128,22.75,7.92,16.0,5,9,10.33,0.93,1,0.2,12,0.2,9925,5587,0.4371,10.11,Basil Pesto,1.78,0.004848,0,0.2215,12,0.086,3,0.6667,5.0,2,5,0.6,1,1.5,0.67,1,2,3,0.6667,0.6,0.0667,2,2,0,3,12,30.0,7.25,1.32,9.87,1.1,6,1,0.1277,0.1915,0,0,2,3,0,0,0.125,1,1,0.125,3,0.125,0.0,0.0,1
155498,25228,7,26,1642368,1,0,0,0,0,1,reorder,0,67,1333,251,19.9,0.8117,5.41,3.42,14.78,6,16,20.67,0.77,2,0.2985,16,0.209,25,14,0.44,9.8,Chai Spice Almond Milk,1.79,1.2E-5,2,0.24,16,0.2,3,0.6667,15.33,1,7,0.0448,61,3.0,20.33,0,0,0,0.0,0.0,-0.0448,2,0,0,3,16,1.0,-4.41,0.18,20.29,1.04,68,1,0.0645,0.3788,0,0,1,1,0,0,0.1714,1,1,0.0,2,0.11426666666666666,0.0,0.0,1
163872,5621,13,104,2760870,1,0,0,0,0,1,reorder,0,17,224,168,13.18,0.25,10.25,12.06,14.65,5,20,13.0,-0.18,3,0.3529,17,0.1765,3003,2509,0.1645,9.61,Sea Salt Fine Crystals,1.2,0.001467,0,0.1931,13,0.0879,1,0.0,4.0,13,13,0.0588,5,0.0,0.0,0,0,1,0.0,0.2,-0.0588,1,0,0,3,18,26.0,15.75,2.54,13.09,0.99,18,1,0.0625,0.2054,1,0,0,3,1,5,0.069,0,1,0.0,1,0.023,0.0,0.0,0
73363,47141,7,77,1180822,1,0,0,0,0,1,reorder,0,23,269,230,11.7,0.145,8.41,9.92,13.0,7,4,9.0,-2.7,4,0.3043,13,0.1304,7498,2491,0.6678,6.02,Cola,3.01,0.003662,1,0.1696,11,0.0915,2,0.5,6.0,8,15,0.087,9,7.0,1.29,0,0,0,0.0,0.0,-0.087,1,0,0,0,19,1.0,-7.41,0.12,10.35,0.77,24,1,0.0669,0.4164,0,0,3,1,6,8,0.0,0,1,0.0,1,0.0,0.0,0.0,1


## 4. Split Customers into Training and Validation

We split customers into:

- 80% training
- 20% validation

The split is done by `user_id`, so the same customer cannot appear in both datasets.

The customer split is saved as a Delta table.

If the notebook is run again, we reuse the saved split instead of creating a new one. This guarantees that every model is evaluated on exactly the same customers.

In [0]:
split_table = "workspace.ml_data.next_basket_user_split"

# Use the existing split if it has already been created
if spark.catalog.tableExists(split_table):

    user_split_df = spark.table(split_table)

    print("Existing customer split loaded")

else:

    # Get one row per customer
    users_df = features_df.select("user_id").distinct()

    # Create the split only once
    train_users_df, validation_users_df = users_df.randomSplit(
        [0.8, 0.2],
        seed=42
    )

    # Add the split name
    train_split_df = (
        train_users_df
        .withColumn(
            "dataset_split",
            F.lit("training")
        )
    )

    validation_split_df = (
        validation_users_df
        .withColumn(
            "dataset_split",
            F.lit("validation")
        )
    )

    # Combine both groups
    user_split_df = train_split_df.unionByName(
        validation_split_df
    )

    # Save permanently
    (
        user_split_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(split_table)
    )

    print("New customer split created and saved")


# Get training customers
train_users_df = (
    user_split_df
    .filter(F.col("dataset_split") == "training")
    .select("user_id")
)

# Get validation customers
validation_users_df = (
    user_split_df
    .filter(F.col("dataset_split") == "validation")
    .select("user_id")
)

# Create the final training dataset
train_df = features_df.join(
    train_users_df,
    on="user_id",
    how="inner"
)

# Create the final validation dataset
validation_df = features_df.join(
    validation_users_df,
    on="user_id",
    how="inner"
)

print("Training and validation datasets ready")

Existing customer split loaded
Training and validation datasets ready


## 5. Validate the Train and Validation Split

We check that:

- No customer appears in both datasets
- Training and validation contain the expected number of customers
- The target distribution is similar in both datasets

In [0]:
# Check customer overlap
overlap_count = (
    train_users_df
    .join(
        validation_users_df,
        on="user_id",
        how="inner"
    )
    .count()
)

print("Customers appearing in both datasets:", overlap_count)

assert overlap_count == 0, "A customer appears in both training and validation."

Customers appearing in both datasets: 0


In [0]:
def target_summary(df, dataset_name):
    total = df.count()
    positives = df.filter(F.col("target_purchased") == 1).count()
    negatives = total - positives

    return (
        dataset_name,
        total,
        positives,
        negatives,
        round(positives / total * 100, 2)
    )

target_distribution = [
    target_summary(train_df, "Training"),
    target_summary(validation_df, "Validation")
]

display(
    spark.createDataFrame(
        target_distribution,
        [
            "dataset",
            "total_rows",
            "positive_rows",
            "negative_rows",
            "positive_percentage"
        ]
    )
)

dataset,total_rows,positive_rows,negative_rows,positive_percentage
Training,10804640,699721,10104919,6.48
Validation,2715125,174199,2540926,6.42


## 6. Select the Model Features

Not every column should be given to the machine learning model.

We remove columns that are only identifiers, text, or the value we want to predict.

We keep `aisle_id` and `department_id` as categorical features.

All other useful numeric columns will be used as model features.

In [0]:
excluded_columns = [
    "user_id",
    "product_id",
    "target_order_id",
    "product_name",
    "candidate_source",
    "target_purchased"
]

categorical_columns = [
    "aisle_id",
    "department_id"
]

numerical_columns = [
    column
    for column in features_df.columns
    if column not in excluded_columns + categorical_columns
]

print("Numerical features:", len(numerical_columns))
print("Categorical features:", len(categorical_columns))
print("Total model features:", len(numerical_columns) + len(categorical_columns))

print("\nCategorical features:")
for column in categorical_columns:
    print("-", column)

Numerical features: 76
Categorical features: 2
Total model features: 78

Categorical features:
- aisle_id
- department_id


## 7. Create or Load the Feature Preparation Pipeline

Spark ML expects the model inputs to be combined into one `features` vector.

We:

- Encode `aisle_id`
- Encode `department_id`
- Combine the categorical and numerical features
- Save the fitted preparation pipeline
- Reuse the saved pipeline if the notebook is run again

This avoids fitting the same preparation steps every time.

In [0]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.functions import vector_to_array

feature_pipeline_path = (
    "/Volumes/workspace/ml_data/models/"
    "instacart_next_basket_feature_pipeline"
)

def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


if path_exists(feature_pipeline_path):

    feature_pipeline_model = PipelineModel.load(
        feature_pipeline_path
    )

    print("Existing feature pipeline loaded")

else:

    aisle_indexer = StringIndexer(
        inputCol="aisle_id",
        outputCol="aisle_id_index",
        handleInvalid="keep"
    )

    department_indexer = StringIndexer(
        inputCol="department_id",
        outputCol="department_id_index",
        handleInvalid="keep"
    )

    encoder = OneHotEncoder(
        inputCols=[
            "aisle_id_index",
            "department_id_index"
        ],
        outputCols=[
            "aisle_id_encoded",
            "department_id_encoded"
        ],
        handleInvalid="keep"
    )

    assembler = VectorAssembler(
        inputCols=numerical_columns + [
            "aisle_id_encoded",
            "department_id_encoded"
        ],
        outputCol="features"
    )

    feature_pipeline = Pipeline(
        stages=[
            aisle_indexer,
            department_indexer,
            encoder,
            assembler
        ]
    )

    feature_pipeline_model = feature_pipeline.fit(
        train_df
    )

    (
        feature_pipeline_model
        .write()
        .overwrite()
        .save(feature_pipeline_path)
    )

    print("Feature pipeline created and saved")


# Apply the same preparation to both datasets
train_ml_df = feature_pipeline_model.transform(
    train_df
)

validation_ml_df = feature_pipeline_model.transform(
    validation_df
)


# Check the final feature-vector size
feature_vector_size = (
    train_ml_df
    .select(
        F.size(
            vector_to_array("features")
        ).alias("feature_vector_size")
    )
    .first()["feature_vector_size"]
)

print("Feature preparation complete")
print("Feature vector size:", feature_vector_size)

Existing feature pipeline loaded
Feature preparation complete
Feature vector size: 233


## 8. Train or Reuse Logistic Regression with MLflow

Logistic Regression is our fast baseline model.

The first time this step is run, we:

- Train the model
- Evaluate ROC AUC and PR AUC
- Save the results in MLflow
- Save the trained model
- Save the validation predictions

If the model and predictions already exist, we load them instead of training the model again.

This makes the notebook faster and allows it to continue after a Databricks session restart.

In [0]:
from pyspark.ml.classification import (
    LogisticRegression,
    LogisticRegressionModel
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
import mlflow


# Saved outputs
lr_model_path = (
    "/Volumes/workspace/ml_data/models/"
    "instacart_next_basket_logistic_regression"
)

lr_predictions_table = (
    "workspace.ml_data."
    "next_basket_lr_validation_predictions"
)


# Keep False during normal notebook runs
RETRAIN_LR = False


# Check whether the previous results already exist
lr_model_exists = path_exists(lr_model_path)

lr_predictions_exist = spark.catalog.tableExists(
    lr_predictions_table
)


should_train_lr = (
    RETRAIN_LR
    or not lr_model_exists
    or not lr_predictions_exist
)


if should_train_lr:

    print("Training Logistic Regression...")

    # Create model
    lr = LogisticRegression(
        featuresCol="features",
        labelCol="target_purchased",
        maxIter=20,
        regParam=0.01,
        elasticNetParam=0.0
    )


    # Evaluators
    roc_evaluator = BinaryClassificationEvaluator(
        labelCol="target_purchased",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    pr_evaluator = BinaryClassificationEvaluator(
        labelCol="target_purchased",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR"
    )


    # Start MLflow run
    with mlflow.start_run(
        run_name="logistic_regression_baseline"
    ) as run:

        # Save model settings
        mlflow.log_params({
            "model_type": "Logistic Regression",
            "maxIter": 20,
            "regParam": 0.01,
            "elasticNetParam": 0.0,
            "feature_vector_size": feature_vector_size,
            "feature_table":
                "workspace.ml_data.next_basket_features",
            "split_table":
                "workspace.ml_data.next_basket_user_split"
        })


        # Train
        lr_model = lr.fit(train_ml_df)


        # Predict
        lr_predictions_full_df = lr_model.transform(
            validation_ml_df
        )


        # Evaluate
        roc_auc = roc_evaluator.evaluate(
            lr_predictions_full_df
        )

        pr_auc = pr_evaluator.evaluate(
            lr_predictions_full_df
        )


        # Save metrics in MLflow
        mlflow.log_metrics({
            "roc_auc": roc_auc,
            "pr_auc": pr_auc
        })


        # Save trained model
        (
            lr_model
            .write()
            .overwrite()
            .save(lr_model_path)
        )


        # Keep only the columns needed later
        lr_predictions_df = (
            lr_predictions_full_df
            .withColumn(
                "purchase_probability",
                vector_to_array("probability")[1]
            )
            .select(
                "user_id",
                "target_order_id",
                "product_id",
                "target_purchased",
                "is_new_to_customer",
                "candidate_source",
                "purchase_probability"
            )
        )


        # Save validation predictions
        (
            lr_predictions_df
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(lr_predictions_table)
        )


        print("Logistic Regression training completed")
        print("Run ID:", run.info.run_id)
        print("ROC AUC:", round(roc_auc, 4))
        print("PR AUC:", round(pr_auc, 4))
        print("Model saved:", lr_model_path)
        print("Predictions saved:", lr_predictions_table)


else:

    # Reuse previous work
    lr_model = LogisticRegressionModel.load(
        lr_model_path
    )

    lr_predictions_df = spark.table(
        lr_predictions_table
    )

    print("Existing Logistic Regression model loaded")
    print("Existing validation predictions loaded")
    print("Model:", lr_model_path)
    print("Predictions:", lr_predictions_table)

Existing Logistic Regression model loaded
Existing validation predictions loaded
Model: /Volumes/workspace/ml_data/models/instacart_next_basket_logistic_regression
Predictions: workspace.ml_data.next_basket_lr_validation_predictions


## 9. Optional Gradient-Boosted Trees Benchmark

We also tested Gradient-Boosted Trees to compare a more complex model with Logistic Regression.

GBT achieved slightly better classification results, but training was much slower.

Because the experiment has already been completed and saved in MLflow, we do not retrain GBT during normal notebook runs.

To intentionally run the benchmark again, set:

`RUN_GBT_BENCHMARK = True`

In [0]:
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Keep False during normal notebook runs
RUN_GBT_BENCHMARK = False


if RUN_GBT_BENCHMARK:

    print("Starting GBT benchmark...")
    print("This model can take a long time to train.")

    # Create evaluators
    gbt_roc_evaluator = BinaryClassificationEvaluator(
        labelCol="target_purchased",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    gbt_pr_evaluator = BinaryClassificationEvaluator(
        labelCol="target_purchased",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR"
    )

    # Create model
    gbt = GBTClassifier(
        featuresCol="features",
        labelCol="target_purchased",
        maxIter=30,
        maxDepth=5,
        stepSize=0.1,
        seed=42
    )

    # Track experiment in MLflow
    with mlflow.start_run(
        run_name="gradient_boosted_trees"
    ) as run:

        mlflow.log_params({
            "model_type": "Gradient-Boosted Trees",
            "maxIter": 30,
            "maxDepth": 5,
            "stepSize": 0.1,
            "seed": 42,
            "feature_vector_size": feature_vector_size,
            "feature_table":
                "workspace.ml_data.next_basket_features",
            "split_table":
                "workspace.ml_data.next_basket_user_split"
        })

        # Train model
        gbt_model = gbt.fit(
            train_ml_df
        )

        # Predictions
        gbt_predictions = gbt_model.transform(
            validation_ml_df
        )

        # Evaluate
        gbt_roc_auc = gbt_roc_evaluator.evaluate(
            gbt_predictions
        )

        gbt_pr_auc = gbt_pr_evaluator.evaluate(
            gbt_predictions
        )

        # Save metrics
        mlflow.log_metrics({
            "roc_auc": gbt_roc_auc,
            "pr_auc": gbt_pr_auc
        })

        print("GBT MLflow run completed")
        print("Run ID:", run.info.run_id)
        print("ROC AUC:", round(gbt_roc_auc, 4))
        print("PR AUC:", round(gbt_pr_auc, 4))

else:

    print("GBT benchmark skipped")
    print("Existing GBT results are already available in MLflow")

GBT benchmark skipped
Existing GBT results are already available in MLflow


## 10. Compare Model Results

The Logistic Regression and Gradient-Boosted Trees experiments are stored in MLflow.

We compare their classification performance before selecting the model used for recommendation scoring.

The main metrics are:

- ROC AUC
- PR AUC
- Training time

In [0]:
from mlflow import MlflowClient

client = MlflowClient()

experiment = mlflow.get_experiment_by_name(
    "/Shared/instacart_next_basket_prediction"
)

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["attributes.start_time DESC"],
    max_results=100
)

wanted_models = [
    "logistic_regression_baseline",
    "gradient_boosted_trees"
]

latest_runs = {}

# Keep only the latest run for each model
for run in runs:

    run_name = run.data.tags.get("mlflow.runName")

    if (
        run_name in wanted_models
        and run_name not in latest_runs
    ):
        latest_runs[run_name] = run


comparison_rows = []

for run_name, run in latest_runs.items():

    # Calculate duration in minutes
    duration_minutes = None

    if (
        run.info.start_time is not None
        and run.info.end_time is not None
    ):
        duration_minutes = round(
            (run.info.end_time - run.info.start_time)
            / 60000,
            2
        )

    comparison_rows.append(
        (
            run_name,
            run.data.metrics.get("roc_auc"),
            run.data.metrics.get("pr_auc"),
            duration_minutes
        )
    )


comparison_df = spark.createDataFrame(
    comparison_rows,
    [
        "model",
        "roc_auc",
        "pr_auc",
        "training_time_minutes"
    ]
)

display(
    comparison_df.orderBy(
        F.col("pr_auc").desc()
    )
)

model,roc_auc,pr_auc,training_time_minutes
gradient_boosted_trees,0.8675979156772964,0.3965787331404492,33.29
logistic_regression_baseline,0.8649537724681906,0.38702210292808614,2.45


## 11. Model Decision and Conclusion

Both models perform well on the next-basket classification task.

Gradient-Boosted Trees gives the best classification results:

- ROC AUC: 0.8676
- PR AUC: 0.3966

Logistic Regression gives slightly lower results:

- ROC AUC: 0.8650
- PR AUC: 0.3870

However, Logistic Regression trains much faster:

- Logistic Regression: about 2.45 minutes
- Gradient-Boosted Trees: about 33.29 minutes

The improvement from GBT is relatively small compared with the additional training time.

For this project, we therefore use Logistic Regression as the scoring model for the recommendation stage.

The trained Logistic Regression model and its validation predictions have already been saved and can be reused without retraining.

The next notebook will use these saved predictions to create and evaluate the final Top 5 recommendation strategy.